In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import json
from pathlib import Path
import warnings
import random
from datetime import datetime
import pandas as pd
import pickle

warnings.filterwarnings('ignore')

# ============================================
# 1. НАСТРОЙКА И КОНФИГУРАЦИЯ
# ============================================

class Config:
    """Конфигурация эксперимента с реальными данными"""
    def __init__(self):
        # Основные параметры
        self.project_name = "cifar10_real_experiment"
        self.task = "multi_class_classification"
        self.dataset = "CIFAR10"
        self.num_classes = 10
        self.class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                           'dog', 'frog', 'horse', 'ship', 'truck']
        self.seed = 42

        # Выбор архитектуры (можно менять)
        self.architecture = "resnet18"  # Варианты: resnet18, resnet34, vgg16, alexnet, custom_cnn

        # Параметры обучения
        self.epochs = 50  # Увеличено для лучших результатов
        self.batch_size = 128
        self.learning_rate = 0.1
        self.momentum = 0.9
        self.weight_decay = 5e-4
        self.patience = 10  # Ранняя остановка

        # Размеры выборок (РЕАЛЬНЫЕ данные CIFAR-10)
        self.train_size = 45000  # 45000 из 50000 для обучения
        self.val_size = 5000     # 5000 для валидации
        self.test_size = 10000   # ВСЕ тестовые данные

        # Аугментации
        self.use_augmentation = True
        self.use_autoaugment = False

        # Пути и файлы
        self.data_dir = "./data"
        self.download = True

        # Устройство
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.device_str = str(self.device)

    def save(self, path="./configs/real_config.json"):
        """Сохранение конфигурации"""
        Path("./configs").mkdir(exist_ok=True)
        config_dict = {
            k: v for k, v in self.__dict__.items()
            if not callable(v) and not k.startswith('_')
        }
        config_dict['device'] = self.device_str

        with open(path, 'w') as f:
            json.dump(config_dict, f, indent=2)

# ============================================
# 2. ДАТАСЕТ С РЕАЛЬНЫМИ ДАННЫМИ
# ============================================

class RealDataManager:
    """Менеджер реальных данных CIFAR-10"""

    def __init__(self, config):
        self.config = config
        self.set_seed()

    def set_seed(self):
        """Установка seed для воспроизводимости"""
        random.seed(self.config.seed)
        np.random.seed(self.config.seed)
        torch.manual_seed(self.config.seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(self.config.seed)
            torch.cuda.manual_seed_all(self.config.seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

    def get_transforms(self, train=True):
        """Трансформации для тренировочных и тестовых данных"""
        if train and self.config.use_augmentation:
            return transforms.Compose([
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.4914, 0.4822, 0.4465],
                    std=[0.2023, 0.1994, 0.2010]
                )
            ])
        else:
            return transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.4914, 0.4822, 0.4465],
                    std=[0.2023, 0.1994, 0.2010]
                )
            ])

    def load_real_data(self):
        """Загрузка реальных данных CIFAR-10"""
        print(" Загрузка реальных данных CIFAR-10...")

        # Тренировочные данные с аугментацией
        train_dataset = datasets.CIFAR10(
            root=self.config.data_dir,
            train=True,
            download=self.config.download,
            transform=self.get_transforms(train=True)
        )

        # Тестовые данные без аугментации
        test_dataset = datasets.CIFAR10(
            root=self.config.data_dir,
            train=False,
            download=self.config.download,
            transform=self.get_transforms(train=False)
        )

        # Разделение на train/val
        indices = list(range(len(train_dataset)))
        np.random.shuffle(indices)

        train_indices = indices[:self.config.train_size]
        val_indices = indices[self.config.train_size:self.config.train_size + self.config.val_size]

        train_subset = Subset(train_dataset, train_indices)
        val_subset = Subset(train_dataset, val_indices)

        # Сохраняем статистику
        self.dataset_stats = {
            'train_size': len(train_subset),
            'val_size': len(val_subset),
            'test_size': len(test_dataset),
            'total_images': len(train_subset) + len(val_subset) + len(test_dataset)
        }

        print(f" РЕАЛЬНЫЕ данные загружены:")
        print(f"   - Тренировочные: {len(train_subset)} изображений")
        print(f"   - Валидационные: {len(val_subset)} изображений")
        print(f"   - Тестовые: {len(test_dataset)} изображений")
        print(f"   - Всего: {self.dataset_stats['total_images']} изображений")

        # Создаем DataLoader'ы
        train_loader = DataLoader(
            train_subset,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=2,
            pin_memory=True
        )

        val_loader = DataLoader(
            val_subset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=2,
            pin_memory=True
        )

        return train_loader, val_loader, test_loader, train_subset, val_subset, test_dataset

    def show_data_statistics(self, dataset, name="Тренировочный набор"):
        """Показать статистику данных"""
        # Собираем метки
        if hasattr(dataset, 'dataset'):
            # Если это Subset
            all_labels = [dataset.dataset.targets[i] for i in dataset.indices]
        else:
            all_labels = dataset.targets

        # Подсчет по классам
        class_counts = {self.config.class_names[i]: 0 for i in range(self.config.num_classes)}
        for label in all_labels:
            class_counts[self.config.class_names[label]] += 1

        print(f"\n Статистика {name}:")
        print("-" * 40)
        for class_name, count in class_counts.items():
            percentage = (count / len(all_labels)) * 100
            print(f"  {class_name:12s}: {count:5d} ({percentage:5.1f}%)")

        # Визуализация распределения
        plt.figure(figsize=(10, 5))
        bars = plt.bar(range(len(class_counts)), list(class_counts.values()))
        plt.xlabel('Классы')
        plt.ylabel('Количество изображений')
        plt.title(f'Распределение классов в {name}')
        plt.xticks(range(len(class_counts)), list(class_counts.keys()), rotation=45, ha='right')

        # Добавляем значения на столбцы
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}', ha='center', va='bottom')

        plt.tight_layout()
        plt.savefig(f'./plots/{name}_distribution.png', dpi=120, bbox_inches='tight')
        plt.close()

        print(f"  Всего: {len(all_labels)} изображений")

# ============================================
# 3. РЕАЛЬНЫЕ МОДЕЛИ АРХИТЕКТУР
# ============================================

class RealModelFactory:
    """Фабрика для создания реальных моделей"""

    @staticmethod
    def create_model(config):
        """Создание модели выбранной архитектуры"""

        if config.architecture == "resnet18":
            model = models.resnet18(pretrained=False)
            # Адаптация для CIFAR-10
            model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            model.maxpool = nn.Identity()
            model.fc = nn.Linear(512, config.num_classes)

        elif config.architecture == "resnet34":
            model = models.resnet34(pretrained=False)
            model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
            model.maxpool = nn.Identity()
            model.fc = nn.Linear(512, config.num_classes)

        elif config.architecture == "vgg16":
            model = models.vgg16(pretrained=False)
            model.features[0] = nn.Conv2d(3, 64, kernel_size=3, padding=1)
            model.classifier[6] = nn.Linear(4096, config.num_classes)

        elif config.architecture == "alexnet":
            model = models.alexnet(pretrained=False)
            model.features[0] = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
            model.classifier[6] = nn.Linear(4096, config.num_classes)

        elif config.architecture == "custom_cnn":
            model = CustomCNN(config.num_classes)

        else:
            raise ValueError(f"Неизвестная архитектура: {config.architecture}")

        print(f" Создана модель: {config.architecture}")
        print(f"   Параметров: {sum(p.numel() for p in model.parameters()):,}")

        return model

class CustomCNN(nn.Module):
    """Кастомная CNN архитектура для CIFAR-10"""
    def __init__(self, num_classes=10):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(
            # Блок 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Блок 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Блок 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# ============================================
# 4. ТРЕНЕР С РЕАЛЬНЫМИ МЕТРИКАМИ
# ============================================

class RealTrainer:
    """Тренер с расчетом реальных метрик"""

    def __init__(self, model, config):
        self.model = model.to(config.device)
        self.config = config
        self.device = config.device

        # История обучения
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.learning_rates = []

        # Лучшие результаты
        self.best_accuracy = 0.0
        self.best_epoch = 0
        self.best_model_state = None

        # Для метрик
        self.all_predictions = []
        self.all_targets = []
        self.metrics = {}

    def train_epoch(self, train_loader, optimizer, criterion, epoch):
        """Обучение на одной эпохе"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0

        pbar = tqdm(train_loader, desc=f'Эпоха {epoch+1}/{self.config.epochs}')
        for batch_idx, (data, target) in enumerate(pbar):
            data, target = data.to(self.device), target.to(self.device)

            optimizer.zero_grad()
            output = self.model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            if batch_idx % 100 == 0:
                acc = 100. * correct / total
                pbar.set_postfix({
                    'Loss': f'{total_loss/(batch_idx+1):.3f}',
                    'Acc': f'{acc:.1f}%'
                })

        epoch_loss = total_loss / len(train_loader)
        epoch_accuracy = 100. * correct / total

        self.train_losses.append(epoch_loss)
        self.train_accuracies.append(epoch_accuracy)

        return epoch_loss, epoch_accuracy

    def validate(self, val_loader, criterion, epoch):
        """Валидация на одной эпохе"""
        self.model.eval()
        val_loss = 0
        correct = 0
        total = 0

        predictions = []
        targets = []

        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                val_loss += criterion(output, target).item()

                _, predicted = output.max(1)
                total += target.size(0)
                correct += predicted.eq(target).sum().item()

                predictions.extend(predicted.cpu().numpy())
                targets.extend(target.cpu().numpy())

        val_loss /= len(val_loader)
        val_accuracy = 100. * correct / total

        self.val_losses.append(val_loss)
        self.val_accuracies.append(val_accuracy)

        self.all_predictions = predictions
        self.all_targets = targets

        # Сохраняем лучшую модель
        if val_accuracy > self.best_accuracy:
            self.best_accuracy = val_accuracy
            self.best_epoch = epoch
            self.best_model_state = self.model.state_dict().copy()
            print(f" Новая лучшая точность: {val_accuracy:.2f}% (эпоха {epoch+1})")

        return val_loss, val_accuracy

    def test(self, test_loader):
        """Тестирование модели"""
        if self.best_model_state:
            self.model.load_state_dict(self.best_model_state)

        self.model.eval()
        test_correct = 0
        test_total = 0
        predictions = []
        targets = []

        with torch.no_grad():
            for data, target in tqdm(test_loader, desc="Тестирование"):
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                _, predicted = output.max(1)
                test_total += target.size(0)
                test_correct += predicted.eq(target).sum().item()

                predictions.extend(predicted.cpu().numpy())
                targets.extend(target.cpu().numpy())

        test_accuracy = 100. * test_correct / test_total
        self.all_predictions = predictions
        self.all_targets = targets

        print(f" Точность на тесте: {test_accuracy:.2f}%")
        return test_accuracy

    def calculate_all_metrics(self):
        """Расчет всех требуемых метрик"""
        y_true = self.all_targets
        y_pred = self.all_predictions

        if len(y_true) == 0 or len(y_pred) == 0:
            print(" Нет данных для расчета метрик")
            return {}

        # Основные метрики (по всем классам)
        accuracy = accuracy_score(y_true, y_pred)
        precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
        recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

        # Weighted метрики
        precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        recall_weighted = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        # По каждому классу
        precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0)
        recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0)
        f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)

        # Матрица ошибок
        cm = confusion_matrix(y_true, y_pred)

        # Детальный отчет
        report = classification_report(
            y_true, y_pred,
            target_names=self.config.class_names,
            output_dict=True,
            zero_division=0
        )

        # Сохраняем метрики
        self.metrics = {
            'overall': {
                'accuracy': float(accuracy),
                'precision_macro': float(precision_macro),
                'recall_macro': float(recall_macro),
                'f1_macro': float(f1_macro),
                'precision_weighted': float(precision_weighted),
                'recall_weighted': float(recall_weighted),
                'f1_weighted': float(f1_weighted)
            },
            'per_class': {},
            'confusion_matrix': cm.tolist(),
            'classification_report': report
        }

        # Заполняем метрики по классам
        for i, class_name in enumerate(self.config.class_names):
            self.metrics['per_class'][class_name] = {
                'precision': float(precision_per_class[i]),
                'recall': float(recall_per_class[i]),
                'f1_score': float(f1_per_class[i]),
                'support': int(np.sum(np.array(y_true) == i))
            }

        return self.metrics

    def save_model(self, filename):
        """Сохранение обученной модели"""
        Path("./models").mkdir(exist_ok=True)
        save_path = f"./models/{filename}"

        torch.save({
            'model_state_dict': self.best_model_state,
            'config': self.config.__dict__,
            'best_accuracy': self.best_accuracy,
            'best_epoch': self.best_epoch,
            'metrics': self.metrics,
            'train_history': {
                'losses': self.train_losses,
                'accuracies': self.train_accuracies
            },
            'val_history': {
                'losses': self.val_losses,
                'accuracies': self.val_accuracies
            }
        }, save_path)

        print(f" Модель сохранена: {save_path}")
        return save_path

# ============================================
# 5. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ============================================

class ResultVisualizer:
    """Класс для визуализации реальных результатов"""

    @staticmethod
    def plot_training_history(trainer, config):
        """Графики обучения"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # График потерь
        ax = axes[0, 0]
        epochs = range(1, len(trainer.train_losses) + 1)
        ax.plot(epochs, trainer.train_losses, 'b-', label='Train', linewidth=2)
        ax.plot(epochs, trainer.val_losses, 'r-', label='Val', linewidth=2)
        ax.set_xlabel('Эпоха')
        ax.set_ylabel('Loss')
        ax.set_title('Кривые обучения: Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # График точности
        ax = axes[0, 1]
        ax.plot(epochs, trainer.train_accuracies, 'b-', label='Train', linewidth=2)
        ax.plot(epochs, trainer.val_accuracies, 'r-', label='Val', linewidth=2)
        ax.axhline(y=trainer.best_accuracy, color='g', linestyle='--',
                  label=f'Best Val: {trainer.best_accuracy:.2f}%')
        ax.set_xlabel('Эпоха')
        ax.set_ylabel('Точность (%)')
        ax.set_title('Кривые обучения: Accuracy')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Матрица ошибок
        ax = axes[1, 0]
        if hasattr(trainer, 'metrics') and 'confusion_matrix' in trainer.metrics:
            cm = np.array(trainer.metrics['confusion_matrix'])
            im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
            ax.figure.colorbar(im, ax=ax)

            # Отображение значений
            thresh = cm.max() / 2.
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    ax.text(j, i, format(cm[i, j], 'd'),
                           ha="center", va="center",
                           color="white" if cm[i, j] > thresh else "black")

            ax.set_xlabel('Предсказанный класс')
            ax.set_ylabel('Истинный класс')
            ax.set_title('Матрица ошибок')
            ax.set_xticks(range(len(config.class_names)))
            ax.set_yticks(range(len(config.class_names)))
            ax.set_xticklabels(config.class_names, rotation=45, ha='right')
            ax.set_yticklabels(config.class_names)

        # Метрики по классам
        ax = axes[1, 1]
        if hasattr(trainer, 'metrics') and 'per_class' in trainer.metrics:
            classes = list(trainer.metrics['per_class'].keys())
            precision = [trainer.metrics['per_class'][c]['precision'] for c in classes]
            recall = [trainer.metrics['per_class'][c]['recall'] for c in classes]
            f1 = [trainer.metrics['per_class'][c]['f1_score'] for c in classes]

            x = np.arange(len(classes))
            width = 0.25

            ax.bar(x - width, precision, width, label='Precision', color='skyblue')
            ax.bar(x, recall, width, label='Recall', color='lightgreen')
            ax.bar(x + width, f1, width, label='F1-Score', color='salmon')

            ax.set_xlabel('Классы')
            ax.set_ylabel('Значение метрики')
            ax.set_title('Метрики по классам')
            ax.set_xticks(x)
            ax.set_xticklabels(classes, rotation=45, ha='right')
            ax.legend()
            ax.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        Path("./plots").mkdir(exist_ok=True)
        plt.savefig('./plots/training_history.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(" Графики обучения сохранены в ./plots/training_history.png")

    @staticmethod
    def create_metrics_table(trainer, config):
        """Создание таблицы с метриками"""
        if not hasattr(trainer, 'metrics') or 'per_class' not in trainer.metrics:
            return None

        # Создаем DataFrame
        data = []
        for class_name, metrics in trainer.metrics['per_class'].items():
            data.append({
                'Класс': class_name,
                'Precision': f"{metrics['precision']:.4f}",
                'Recall': f"{metrics['recall']:.4f}",
                'F1-Score': f"{metrics['f1_score']:.4f}",
                'Support': metrics['support']
            })

        # Добавляем общие метрики
        if 'overall' in trainer.metrics:
            data.append({
                'Класс': 'Overall (Macro)',
                'Precision': f"{trainer.metrics['overall']['precision_macro']:.4f}",
                'Recall': f"{trainer.metrics['overall']['recall_macro']:.4f}",
                'F1-Score': f"{trainer.metrics['overall']['f1_macro']:.4f}",
                'Support': 'N/A'
            })
            data.append({
                'Класс': 'Overall (Weighted)',
                'Precision': f"{trainer.metrics['overall']['precision_weighted']:.4f}",
                'Recall': f"{trainer.metrics['overall']['recall_weighted']:.4f}",
                'F1-Score': f"{trainer.metrics['overall']['f1_weighted']:.4f}",
                'Support': 'N/A'
            })
            data.append({
                'Класс': 'Accuracy',
                'Precision': f"{trainer.metrics['overall']['accuracy']:.4f}",
                'Recall': 'N/A',
                'F1-Score': 'N/A',
                'Support': 'N/A'
            })

        df = pd.DataFrame(data)

        # Сохраняем
        Path("./results").mkdir(exist_ok=True)
        df.to_csv('./results/metrics_table.csv', index=False, encoding='utf-8')
        df.to_excel('./results/metrics_table.xlsx', index=False)

        print(" Таблица метрик сохранена в ./results/")
        return df

    @staticmethod
    def save_detailed_report(trainer, config, test_accuracy):
        """Сохранение детального отчета"""
        report = {
            'experiment_info': {
                'project_name': config.project_name,
                'architecture': config.architecture,
                'dataset': config.dataset,
                'timestamp': datetime.now().isoformat(),
                'device': config.device_str
            },
            'training_info': {
                'epochs': config.epochs,
                'batch_size': config.batch_size,
                'learning_rate': config.learning_rate,
                'best_val_accuracy': float(trainer.best_accuracy),
                'best_epoch': int(trainer.best_epoch),
                'test_accuracy': float(test_accuracy)
            },
            'metrics': trainer.metrics if hasattr(trainer, 'metrics') else {},
            'training_history': {
                'train_losses': [float(l) for l in trainer.train_losses],
                'val_losses': [float(l) for l in trainer.val_losses],
                'train_accuracies': [float(a) for a in trainer.train_accuracies],
                'val_accuracies': [float(a) for a in trainer.val_accuracies]
            }
        }

        with open('./results/detailed_report.json', 'w', encoding='utf-8') as f:
            json.dump(report, f, indent=2, ensure_ascii=False)

        print(" Детальный отчет сохранен в ./results/detailed_report.json")
        return report

# ============================================
# 6. ОСНОВНОЙ ЭКСПЕРИМЕНТ С РЕАЛЬНЫМИ ДАННЫМИ
# ============================================

def run_real_experiment():
    """Запуск полного эксперимента с реальными данными"""
    print("=" * 70)
    print(" ПОЛНЫЙ ЭКСПЕРИМЕНТ НА РЕАЛЬНЫХ ДАННЫХ CIFAR-10")
    print("=" * 70)

    # Создание конфигурации
    config = Config()
    config.save()

    print(f" Конфигурация эксперимента:")
    print(f"   - Архитектура: {config.architecture}")
    print(f"   - Эпох: {config.epochs}")
    print(f"   - Batch size: {config.batch_size}")
    print(f"   - Устройство: {config.device}")
    print(f"   - Данные: CIFAR-10 (реальные)")

    # Загрузка реальных данных
    data_manager = RealDataManager(config)
    train_loader, val_loader, test_loader, train_subset, val_subset, test_dataset = data_manager.load_real_data()

    # Показать статистику данных
    data_manager.show_data_statistics(train_subset, "Тренировочный набор")
    data_manager.show_data_statistics(val_subset, "Валидационный набор")
    data_manager.show_data_statistics(test_dataset, "Тестовый набор")

    # Создание модели
    model = RealModelFactory.create_model(config)

    # Оптимизатор и функция потерь
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(),
        lr=config.learning_rate,
        momentum=config.momentum,
        weight_decay=config.weight_decay
    )

    # Планировщик learning rate
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)

    # Создание тренера
    trainer = RealTrainer(model, config)

    # Обучение модели
    print(f"\n НАЧАЛО ОБУЧЕНИЯ НА РЕАЛЬНЫХ ДАННЫХ")
    print("-" * 50)

    for epoch in range(config.epochs):
        print(f"\nЭпоха {epoch+1}/{config.epochs}")

        # Обучение
        train_loss, train_acc = trainer.train_epoch(train_loader, optimizer, criterion, epoch)

        # Валидация
        val_loss, val_acc = trainer.validate(val_loader, criterion, epoch)

        # Обновление learning rate
        scheduler.step()

        print(f"   Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"   Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(f"   Best Val Acc: {trainer.best_accuracy:.2f}%")

        # Ранняя остановка
        if epoch - trainer.best_epoch > config.patience and trainer.best_accuracy > 85:
            print(f" Ранняя остановка на эпохе {epoch+1}")
            break

    # Тестирование
    print(f"\n ТЕСТИРОВАНИЕ НА РЕАЛЬНЫХ ДАННЫХ")
    print("-" * 50)

    test_accuracy = trainer.test(test_loader)

    # Расчет метрик
    print(f"\n РАСЧЕТ МЕТРИК")
    print("-" * 50)
    metrics = trainer.calculate_all_metrics()

    # Вывод результатов
    print(f"\n ИТОГОВЫЕ РЕЗУЛЬТАТЫ:")
    print(f"   Лучшая точность (валидация): {trainer.best_accuracy:.2f}%")
    print(f"   Точность на тесте: {test_accuracy:.2f}%")

    if metrics and 'overall' in metrics:
        print(f"\n ОБЩИЕ МЕТРИКИ:")
        print(f"   Accuracy: {metrics['overall']['accuracy']:.4f}")
        print(f"   Precision (macro): {metrics['overall']['precision_macro']:.4f}")
        print(f"   Recall (macro): {metrics['overall']['recall_macro']:.4f}")
        print(f"   F1-Score (macro): {metrics['overall']['f1_macro']:.4f}")

    # Визуализация результатов
    print(f"\n ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ")
    print("-" * 50)
    visualizer = ResultVisualizer()
    visualizer.plot_training_history(trainer, config)
    metrics_table = visualizer.create_metrics_table(trainer, config)
    detailed_report = visualizer.save_detailed_report(trainer, config, test_accuracy)

    # Сохранение модели
    model_path = trainer.save_model(f"{config.architecture}_real_model.pth")

    # Создание финального отчета
    create_final_report(config, trainer, test_accuracy, metrics)

    print("\n" + "=" * 70)
    print(" ЭКСПЕРИМЕНТ НА РЕАЛЬНЫХ ДАННЫХ УСПЕШНО ЗАВЕРШЕН!")
    print("=" * 70)

    return trainer, metrics, test_accuracy

def create_final_report(config, trainer, test_accuracy, metrics):
    """Создание финального отчета в Markdown"""
    Path("./reports").mkdir(exist_ok=True)

    with open('./reports/final_report.md', 'w', encoding='utf-8') as f:
        f.write("# ОТЧЕТ ПО ЭКСПЕРИМЕНТУ КЛАССИФИКАЦИИ CIFAR-10\n\n")
        f.write(f"**Дата проведения:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        f.write("## Конфигурация эксперимента\n")
        f.write("```json\n")
        f.write(json.dumps(config.__dict__, indent=2, default=str))
        f.write("\n```\n\n")

        f.write("## Результаты обучения\n\n")
        f.write(f"- **Архитектура:** {config.architecture}\n")
        f.write(f"- **Лучшая точность на валидации:** {trainer.best_accuracy:.2f}%\n")
        f.write(f"- **Точность на тесте:** {test_accuracy:.2f}%\n")
        f.write(f"- **Эпоха лучшей модели:** {trainer.best_epoch + 1}\n")
        f.write(f"- **Всего эпох обучения:** {len(trainer.train_losses)}\n\n")

        f.write("## Метрики классификации\n\n")

        if metrics and 'overall' in metrics:
            f.write("### Общие метрики\n")
            f.write("| Метрика | Значение |\n")
            f.write("|---------|----------|\n")
            f.write(f"| Accuracy | {metrics['overall']['accuracy']:.4f} |\n")
            f.write(f"| Precision (macro) | {metrics['overall']['precision_macro']:.4f} |\n")
            f.write(f"| Recall (macro) | {metrics['overall']['recall_macro']:.4f} |\n")
            f.write(f"| F1-Score (macro) | {metrics['overall']['f1_macro']:.4f} |\n")
            f.write(f"| Precision (weighted) | {metrics['overall']['precision_weighted']:.4f} |\n")
            f.write(f"| Recall (weighted) | {metrics['overall']['recall_weighted']:.4f} |\n")
            f.write(f"| F1-Score (weighted) | {metrics['overall']['f1_weighted']:.4f} |\n")
            f.write("\n")

        if metrics and 'per_class' in metrics:
            f.write("### Метрики по классам\n")
            f.write("| Класс | Precision | Recall | F1-Score | Support |\n")
            f.write("|-------|-----------|--------|----------|---------|\n")
            for class_name, class_metrics in metrics['per_class'].items():
                f.write(f"| {class_name} | {class_metrics['precision']:.4f} | ")
                f.write(f"{class_metrics['recall']:.4f} | {class_metrics['f1_score']:.4f} | ")
                f.write(f"{class_metrics['support']} |\n")

        f.write("\n## Анализ результатов\n\n")

        if test_accuracy > 85:
            f.write(" **Отличный результат!** Модель достигла высокой точности (>85%) на тестовом наборе.\n")
        elif test_accuracy > 75:
            f.write(" **Хороший результат.** Модель показала удовлетворительную точность.\n")
        else:
            f.write(" **Результат можно улучшить.** Рекомендуется увеличить количество эпох или изменить архитектуру.\n")

        f.write("\n## Сохраненные файлы\n\n")
        f.write("- **Модель:** `./models/{config.architecture}_real_model.pth`\n")
        f.write("- **Метрики:** `./results/metrics_table.csv`\n")
        f.write("- **Графики:** `./plots/training_history.png`\n")
        f.write("- **Детальный отчет:** `./results/detailed_report.json`\n")
        f.write("- **Конфигурация:** `./configs/real_config.json`\n")

        f.write("\n## Рекомендации для улучшения\n\n")
        f.write("1. Увеличить количество эпох обучения\n")
        f.write("2. Использовать более сложную архитектуру (ResNet50, EfficientNet)\n")
        f.write("3. Применить дополнительные аугментации\n")
        f.write("4. Использовать transfer learning с предобученными весами\n")

    print(" Финальный отчет сохранен в ./reports/final_report.md")

# ============================================
# 7. ЗАПУСК ЭКСПЕРИМЕНТА
# ============================================

if __name__ == "__main__":
    # Создание необходимых директорий
    directories = ["data", "configs", "models", "results", "plots", "reports", "logs"]
    for dir_name in directories:
        Path(f"./{dir_name}").mkdir(exist_ok=True)

    print("=" * 70)
    print(" ЗАПУСК ЭКСПЕРИМЕНТА КЛАССИФИКАЦИИ НА РЕАЛЬНЫХ ДАННЫХ")
    print("=" * 70)
    print("Этот код использует РЕАЛЬНЫЕ данные CIFAR-10:")
    print("  - 50,000 тренировочных изображений")
    print("  - 10,000 тестовых изображений")
    print("  - 10 классов объектов")
    print("=" * 70)

    try:
        # Запуск эксперимента
        trainer, metrics, test_accuracy = run_real_experiment()

        # Вывод итогов
        print("\n" + "=" * 70)
        print(" ЭКСПЕРИМЕНТ УСПЕШНО ЗАВЕРШЕН!")
        print("=" * 70)
        print(f"\n РЕЗУЛЬТАТЫ:")
        print(f"  Архитектура: {trainer.config.architecture}")
        print(f"  Лучшая точность: {trainer.best_accuracy:.2f}%")
        print(f"  Тестовая точность: {test_accuracy:.2f}%")

        if metrics and 'overall' in metrics:
            print(f"\n ОСНОВНЫЕ МЕТРИКИ:")
            print(f"  Accuracy: {metrics['overall']['accuracy']:.4f}")
            print(f"  F1-Score: {metrics['overall']['f1_macro']:.4f}")

        print(f"\n СОХРАНЕННЫЕ ФАЙЛЫ:")
        print(f"  - Модель: ./models/{trainer.config.architecture}_real_model.pth")
        print(f"  - Отчет: ./reports/final_report.md")
        print(f"  - Метрики: ./results/metrics_table.csv")
        print(f"  - Графики: ./plots/training_history.png")

    except Exception as e:
        print(f"\n Ошибка при выполнении эксперимента: {e}")
        import traceback
        traceback.print_exc()

 ЗАПУСК ЭКСПЕРИМЕНТА КЛАССИФИКАЦИИ НА РЕАЛЬНЫХ ДАННЫХ
Этот код использует РЕАЛЬНЫЕ данные CIFAR-10:
  - 50,000 тренировочных изображений
  - 10,000 тестовых изображений
  - 10 классов объектов
 ПОЛНЫЙ ЭКСПЕРИМЕНТ НА РЕАЛЬНЫХ ДАННЫХ CIFAR-10
 Конфигурация эксперимента:
   - Архитектура: resnet18
   - Эпох: 50
   - Batch size: 128
   - Устройство: cuda
   - Данные: CIFAR-10 (реальные)
 Загрузка реальных данных CIFAR-10...
 РЕАЛЬНЫЕ данные загружены:
   - Тренировочные: 45000 изображений
   - Валидационные: 5000 изображений
   - Тестовые: 10000 изображений
   - Всего: 60000 изображений

 Статистика Тренировочный набор:
----------------------------------------
  airplane    :  4490 ( 10.0%)
  automobile  :  4513 ( 10.0%)
  bird        :  4498 ( 10.0%)
  cat         :  4526 ( 10.1%)
  deer        :  4476 (  9.9%)
  dog         :  4497 ( 10.0%)
  frog        :  4495 ( 10.0%)
  horse       :  4466 (  9.9%)
  ship        :  4501 ( 10.0%)
  truck       :  4538 ( 10.1%)
  Всего: 45000 изображени

Эпоха 1/50:   0%|          | 0/352 [00:00<?, ?it/s]

 Новая лучшая точность: 42.06% (эпоха 1)
   Train Loss: 1.9203, Train Acc: 30.92%
   Val Loss: 1.5916, Val Acc: 42.06%
   Best Val Acc: 42.06%

Эпоха 2/50


Эпоха 2/50:   0%|          | 0/352 [00:00<?, ?it/s]

 Новая лучшая точность: 53.86% (эпоха 2)
   Train Loss: 1.3703, Train Acc: 49.94%
   Val Loss: 1.3276, Val Acc: 53.86%
   Best Val Acc: 53.86%

Эпоха 3/50


...

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aa3d3f37a60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
Exception ignored in: AssertionError: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aa3d3f37a60>can only test a child process

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aa3d3f37a60>
self._shutdown_workers(

 Новая лучшая точность: 88.28% (эпоха 31)
   Train Loss: 0.1747, Train Acc: 94.04%
   Val Loss: 0.3465, Val Acc: 88.28%
   Best Val Acc: 88.28%

Эпоха 32/50


Эпоха 32/50:   0%|          | 0/352 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7aa3d3f37a60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    Exception ignored in: assert self._parent_pid == os.getpid(), 'can only test a child process'<function _MultiProcessingDataLoaderIter.__del__ at 0x7aa3d3f37a60>
 
    Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1654, in __del__
      self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1637, in _shutdown_workers
 ^    if w.is_alive():^^
 ^^   ^ ^ ^ ^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 